# Ingesta de uso de Microsoft 365 Copilot en Fabric
Demo para **Microsoft 365 Live 2026** · ¿Está funcionando Copilot en tu empresa? Mide la adopción con Fabric

Qué hace este notebook:
1. Obtiene un token de Microsoft Graph con un Service Principal (secretos en Key Vault).
2. Descarga el informe de uso de Copilot por usuario y los datos organizativos de Entra ID.
3. Guarda la respuesta en bruto (`Files/raw`) y **añade la foto del día** al histórico en tablas Delta.
4. Genera las dimensiones `dim_users`, `dim_app` y `dim_date`.

`MODO = "simulado"` genera 12 semanas de datos ficticios con la misma estructura que Graph (para ensayar y para tenants con poco uso).
`MODO = "real"` llama a Graph. Adjunta el Lakehouse `lh_copilot_adoption` como predeterminado antes de ejecutar.

In [ ]:
# Parámetros (celda marcada como 'parameters' para poder sobrescribirlos desde un pipeline)
MODO = "simulado"            # "simulado" | "real"
KEY_VAULT_URL = "https://<tu-keyvault>.vault.azure.net/"
SECRET_TENANT_ID = "copilot-tenant-id"
SECRET_CLIENT_ID = "copilot-client-id"
SECRET_CLIENT_SECRET = "copilot-client-secret"
PERIODO = "D30"              # D7, D30, D90, D180
DIAS_HISTORICO_SIM = 84      # 12 semanas de histórico simulado
USUARIOS_SIM = 120
SEMILLA = 42
DOMINIO_SIM = "contoso.com"

In [ ]:
import io, json, re, time, datetime as dt
import requests
import numpy as np
import pandas as pd

RAW_PATH = "/lakehouse/default/Files/raw/copilot_usage"
HOY = dt.date.today()

# Traducción de columnas de Graph a nombres de app legibles
APP_MAP = {
    "microsoftTeamsCopilotLastActivityDate": "Teams",
    "wordCopilotLastActivityDate": "Word",
    "excelCopilotLastActivityDate": "Excel",
    "powerPointCopilotLastActivityDate": "PowerPoint",
    "outlookCopilotLastActivityDate": "Outlook",
    "oneNoteCopilotLastActivityDate": "OneNote",
    "loopCopilotLastActivityDate": "Loop",
    "copilotChatLastActivityDate": "Copilot Chat",
}
APP_ORDEN = ["Teams", "Outlook", "Word", "Excel", "PowerPoint", "Copilot Chat", "OneNote", "Loop"]
print(f"Modo: {MODO} · fecha de ejecución: {HOY}")

## 1. Token de Microsoft Graph (solo modo real)

In [ ]:
def obtener_token():
    tenant = notebookutils.credentials.getSecret(KEY_VAULT_URL, SECRET_TENANT_ID)
    client = notebookutils.credentials.getSecret(KEY_VAULT_URL, SECRET_CLIENT_ID)
    secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, SECRET_CLIENT_SECRET)
    r = requests.post(
        f"https://login.microsoftonline.com/{tenant}/oauth2/v2.0/token",
        data={"grant_type": "client_credentials", "client_id": client,
              "client_secret": secret, "scope": "https://graph.microsoft.com/.default"},
        timeout=30)
    r.raise_for_status()
    return r.json()["access_token"]

TOKEN = obtener_token() if MODO == "real" else None
print("Token obtenido" if TOKEN else "Modo simulado: no se pide token")

## 2. Llamadas a Graph con paginación y reintentos

In [ ]:
def graph_get(url, token, accept_json=True, reintentos=5):
    """GET a Graph con reintento ante 429/503. Devuelve el objeto requests.Response."""
    headers = {"Authorization": f"Bearer {token}"}
    if accept_json:
        headers["Accept"] = "application/json"
    for intento in range(reintentos):
        r = requests.get(url, headers=headers, timeout=60)
        if r.status_code in (429, 503):
            espera = int(r.headers.get("Retry-After", 2 ** intento))
            print(f"  {r.status_code}, reintento en {espera}s")
            time.sleep(espera)
            continue
        r.raise_for_status()
        return r
    raise RuntimeError(f"Graph no responde tras {reintentos} intentos: {url}")

def graph_get_all(url, token):
    """Recorre @odata.nextLink y devuelve la lista completa de 'value'."""
    filas = []
    while url:
        datos = graph_get(url, token).json()
        filas.extend(datos.get("value", []))
        url = datos.get("@odata.nextLink")
    return filas

def a_camel(nombre):
    """'Microsoft Teams Copilot Last Activity Date' -> 'microsoftTeamsCopilotLastActivityDate'"""
    partes = re.sub(r"[^0-9a-zA-Z ]", " ", nombre).split()
    return partes[0].lower() + "".join(p[:1].upper() + p[1:] for p in partes[1:]) if partes else nombre

In [ ]:
def descargar_uso_copilot(token):
    """Intenta JSON (ruta beta); si falla, CSV (v1.0). Devuelve (DataFrame, texto_bruto)."""
    try:
        url = f"https://graph.microsoft.com/beta/copilot/reports/getMicrosoft365CopilotUsageUserDetail(period='{PERIODO}')?$format=application/json"
        filas = graph_get_all(url, token)
        print(f"JSON (beta): {len(filas)} usuarios")
        return pd.DataFrame(filas), json.dumps(filas)
    except Exception as e:
        print(f"JSON no disponible ({e}); pruebo CSV v1.0")
        url = f"https://graph.microsoft.com/v1.0/copilot/reports/getMicrosoft365CopilotUsageUserDetail(period='{PERIODO}')"
        texto = graph_get(url, token, accept_json=False).content.decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(texto))
        df.columns = [a_camel(c) for c in df.columns]
        print(f"CSV (v1.0): {len(df)} usuarios")
        return df, texto

def descargar_usuarios(token):
    url = ("https://graph.microsoft.com/v1.0/users?$top=999"
           "&$select=userPrincipalName,displayName,department,country,jobTitle,accountEnabled")
    return pd.DataFrame(graph_get_all(url, token))

## 3. Generador de datos simulados (misma estructura que Graph)

In [ ]:
DEPARTAMENTOS = {
    # departamento: (adopción inicial, adopción final a las 12 semanas)
    "IT":          (0.40, 0.90),
    "Marketing":   (0.35, 0.85),
    "Comercial":   (0.20, 0.70),
    "Finanzas":    (0.10, 0.60),
    "Operaciones": (0.05, 0.45),
    "RRHH":        (0.05, 0.20),
}
PESO_APP = {"Teams": 1.0, "Outlook": 0.9, "Copilot Chat": 0.7, "Word": 0.6,
            "Excel": 0.45, "PowerPoint": 0.35, "OneNote": 0.08, "Loop": 0.05}
NOMBRES = ["Ana", "Luis", "Marta", "Jorge", "Lucía", "Pablo", "Carmen", "Sergio", "Elena", "David",
           "Laura", "Javier", "Sara", "Raúl", "Nuria", "Iván", "Paula", "Óscar", "Irene", "Álvaro"]
APELLIDOS = ["García", "Martín", "Ruiz", "López", "Sánchez", "Pérez", "Gómez", "Díaz", "Moreno", "Navarro",
             "Torres", "Romero", "Vidal", "Serra", "Ferrer", "Ortega"]
PAISES = ["España", "España", "España", "México", "Colombia"]

def _sin_tildes(s):
    import unicodedata
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def generar_simulado(n_usuarios=USUARIOS_SIM, dias=DIAS_HISTORICO_SIM, semilla=SEMILLA, hoy=HOY, lag=2):
    rng = np.random.default_rng(semilla)
    deps = list(DEPARTAMENTOS)
    usuarios, vistos = [], set()
    for i in range(n_usuarios):
        while True:
            nom, ape = rng.choice(NOMBRES), rng.choice(APELLIDOS)
            upn = f"{_sin_tildes(nom).lower()}.{_sin_tildes(ape).lower()}@{DOMINIO_SIM}"
            if upn not in vistos:
                vistos.add(upn); break
        usuarios.append({"userPrincipalName": upn, "displayName": f"{nom} {ape}",
                         "department": deps[i % len(deps)], "country": rng.choice(PAISES),
                         "jobTitle": "Empleado", "accountEnabled": True})
    dim_u = pd.DataFrame(usuarios)
    dormidos = set(rng.choice(dim_u.userPrincipalName, size=n_usuarios // 10, replace=False))

    calentamiento = 28                            # 4 semanas previas para que la primera foto ya tenga ventana completa
    inicio = hoy - dt.timedelta(days=dias + lag + calentamiento)
    fechas = [inicio + dt.timedelta(days=d) for d in range(dias + lag + calentamiento + 1)]
    # Umbrales estratificados por departamento: la adopción final se parece de verdad a la del diccionario
    umbrales = {}
    for dep, grupo in dim_u.groupby("department"):
        orden = rng.permutation(len(grupo))
        for k, upn in zip(orden, grupo.userPrincipalName):
            umbrales[upn] = (k + rng.random()) / len(grupo)
    # Eventos de uso: (upn, app, fecha)
    eventos = []
    for _, u in dim_u.iterrows():
        if u.userPrincipalName in dormidos:
            continue
        a0, a1 = DEPARTAMENTOS[u.department]
        umbral = umbrales[u.userPrincipalName]    # cuándo "se engancha" este usuario
        if umbral > a1:
            continue                              # no llega a adoptar en el periodo
        n = len(fechas)
        dia_inicio = 0 if umbral <= a0 else int(n * (umbral - a0) / (a1 - a0))
        dia_fin = n
        if rng.random() < 0.15:                   # 15 % prueba y lo deja
            dia_fin = min(n, dia_inicio + int(rng.integers(10, 26)))
        frecuencia = rng.uniform(0.2, 0.6)        # probabilidad de usarlo un día laborable
        for d in range(dia_inicio, dia_fin):
            f = fechas[d]
            if f.weekday() >= 5 or rng.random() > frecuencia:
                continue
            for app, peso in PESO_APP.items():
                if rng.random() < peso * 0.35:
                    eventos.append((u.userPrincipalName, app, f))
    ev = pd.DataFrame(eventos, columns=["upn", "app", "fecha"])

    # Una foto por día: última actividad conocida a fecha de refresco (foto - lag)
    fotos = []
    inv_map = {v: k for k, v in APP_MAP.items()}
    for d in range(dias):
        foto = hoy - dt.timedelta(days=dias - 1 - d)
        refresco = foto - dt.timedelta(days=lag)
        ult = ev[ev.fecha <= refresco].groupby(["upn", "app"]).fecha.max().unstack()
        snap = dim_u[["userPrincipalName", "displayName"]].copy()
        snap["reportRefreshDate"] = refresco.isoformat()
        for app, col in inv_map.items():
            snap[col] = snap.userPrincipalName.map(ult[app] if app in ult else {}).map(
                lambda x: x.isoformat() if isinstance(x, dt.date) else None)
        cols_app = list(inv_map.values())
        snap["lastActivityDate"] = snap[cols_app].apply(lambda r: max([x for x in r if isinstance(x, str)], default=None), axis=1)
        snap["_snapshot_date"] = foto
        fotos.append(snap)
    return dim_u, pd.concat(fotos, ignore_index=True)

## 4. Transformación: de la respuesta de Graph a tablas del modelo

In [ ]:
def normalizar_uso(df_raw, snapshot_date):
    """Devuelve (copilot_usage_user, copilot_usage_user_app) para una o varias fotos."""
    df = df_raw.copy()
    if "_snapshot_date" not in df:
        df["_snapshot_date"] = snapshot_date
    fecha = lambda s: pd.to_datetime(s, errors="coerce").dt.date
    user = pd.DataFrame({
        "snapshot_date": df["_snapshot_date"],
        "report_refresh_date": fecha(df.get("reportRefreshDate")),
        "upn": df["userPrincipalName"].str.lower(),
        "display_name": df.get("displayName"),
        "last_activity_date": fecha(df.get("lastActivityDate")),
    })
    # Unpivot de todas las columnas *LastActivityDate por app (incluye apps nuevas que añada Microsoft)
    cols_app = [c for c in df.columns if c.endswith("LastActivityDate") and c != "lastActivityDate"]
    largo = df.melt(id_vars=["_snapshot_date", "reportRefreshDate", "userPrincipalName"], value_vars=cols_app,
                    var_name="col", value_name="last_activity_date")
    user_app = pd.DataFrame({
        "snapshot_date": largo["_snapshot_date"],
        "report_refresh_date": fecha(largo["reportRefreshDate"]),
        "upn": largo["userPrincipalName"].str.lower(),
        "app": largo["col"].map(lambda c: APP_MAP.get(c, c.replace("CopilotLastActivityDate", ""))),
        "last_activity_date": fecha(largo["last_activity_date"]),
    })
    return user, user_app

def construir_dim_users(df_users):
    return pd.DataFrame({
        "upn": df_users["userPrincipalName"].str.lower(),
        "display_name": df_users.get("displayName"),
        "department": df_users.get("department").fillna("Sin departamento"),
        "country": df_users.get("country").fillna("Sin país"),
        "job_title": df_users.get("jobTitle"),
    }).drop_duplicates("upn")

def construir_dim_app(apps):
    orden = {a: i + 1 for i, a in enumerate(APP_ORDEN)}
    return pd.DataFrame({"app": sorted(set(apps), key=lambda a: orden.get(a, 99))}).assign(
        orden=lambda d: d.app.map(lambda a: orden.get(a, 99)))

def construir_dim_date(desde, hasta):
    fechas = pd.date_range(desde - dt.timedelta(days=200), hasta, freq="D")
    meses = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
    return pd.DataFrame({
        "date": fechas.date,
        "year": fechas.year,
        "month": fechas.month,
        "month_name": [f"{meses[m-1]} {y}" for m, y in zip(fechas.month, fechas.year)],
        "week_start": (fechas - pd.to_timedelta(fechas.weekday, unit="D")).date,
        "is_weekday": fechas.weekday < 5,
    })

## 5. Ejecución: descarga (o simulación) y guardado en bruto

In [ ]:
if MODO == "real":
    df_uso_raw, texto_bruto = descargar_uso_copilot(TOKEN)
    df_users_raw = descargar_usuarios(TOKEN)
    carpeta = f"{RAW_PATH}/snapshot_date={HOY.isoformat()}"
    notebookutils.fs.mkdirs(carpeta.replace("/lakehouse/default/", ""))
    with open(f"{carpeta}/usage.json", "w", encoding="utf-8") as f:
        f.write(texto_bruto)
    print(f"Guardado en bruto: {carpeta}/usage.json")
else:
    df_users_raw, df_uso_raw = generar_simulado()
    print(f"Simulados {len(df_users_raw)} usuarios y {df_uso_raw._snapshot_date.nunique()} fotos diarias")

uso_user, uso_user_app = normalizar_uso(df_uso_raw, HOY)
dim_users = construir_dim_users(df_users_raw)
dim_app = construir_dim_app(uso_user_app.app.unique())
dim_date = construir_dim_date(uso_user.snapshot_date.min(), HOY)
print(uso_user.shape, uso_user_app.shape, dim_users.shape)

## 6. Escritura en Delta: la foto del día se añade al histórico
En modo real se borra primero la foto de hoy (si ya existía) y se añade: así el notebook se puede relanzar sin duplicar.
En modo simulado se sobrescribe todo el histórico.

In [ ]:
from pyspark.sql import functions as F

def escribir_hechos(pdf, tabla):
    sdf = spark.createDataFrame(pdf)
    if MODO == "real" and spark.catalog.tableExists(tabla):
        spark.sql(f"DELETE FROM {tabla} WHERE snapshot_date = '{HOY.isoformat()}'")
        sdf.write.format("delta").mode("append").saveAsTable(tabla)
    else:
        sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla)
    print(f"{tabla}: {spark.table(tabla).count()} filas")

def escribir_dim(pdf, tabla):
    spark.createDataFrame(pdf).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(tabla)
    print(f"{tabla}: {len(pdf)} filas")

escribir_hechos(uso_user, "copilot_usage_user")
escribir_hechos(uso_user_app, "copilot_usage_user_app")
escribir_dim(dim_users, "dim_users")
escribir_dim(dim_app, "dim_app")
escribir_dim(dim_date, "dim_date")

## 7. Comprobación rápida (la que se enseña en directo)

In [ ]:
display(spark.sql("""
    SELECT u.snapshot_date,
           COUNT(*)                                                         AS licencias,
           SUM(CASE WHEN datediff(u.report_refresh_date, u.last_activity_date) < 28 THEN 1 ELSE 0 END) AS activos_28d,
           ROUND(100 * SUM(CASE WHEN datediff(u.report_refresh_date, u.last_activity_date) < 28 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_adopcion
    FROM copilot_usage_user u
    GROUP BY u.snapshot_date
    ORDER BY u.snapshot_date DESC
    LIMIT 14
"""))